# SEC EDGAR Filing Pipeline

Enter your CIK codes, date range, and form types below, then run the cells in order.

This notebook drives three layers, each in its own module under `methods/`:

1. **Filing metadata** — `sec_filings_sync.fetch_filings_for_ciks` → one `FilingRecord`
   per filing (all the EDGAR URLs, dates, XBRL flag).
2. **Structured XBRL extraction** — `sec_xbrl_extract`: a tidy one-row-per-fact table,
   plus the DEI cover page split into **entity** and **security** levels.
3. **Advanced / manual extraction (NEW)** — `sec_filing_manual_extract`: works off the raw
   **full submission text** (`submission_txt_url`) to
   (a) save a clean, gzip-compressed HTML render of the primary document,
   (b) pull **geographic / revenue / segment / product** tables, and
   (c) flag geographic-**concentration "footnote" statements** such as
   *"Substantially all of our revenue is derived from the United States."*

All SEC traffic — metadata, XBRL, and full submission text — flows through the one
rate-limited request path in `sec_filings_sync` (≤10 req/s, automatic 429 back-off).

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
from dataclasses import asdict

from methods.sec_filings_sync import fetch_filings_for_ciks, SECBlockedError
from methods.sec_xbrl_extract import parse_filings, cover_pages
from methods.sec_filing_manual_extract import (
    process_submission,   # one filing  -> ManualExtraction
    process_filings,      # many filings -> list[ManualExtraction]
    statements_frame,     # flatten concentration statements across filings
    tables_frame,         # flatten matched-table catalog across filings
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

## 1. Inputs

In [ ]:
# Add or remove CIK codes here (strings or ints both work)
CIKS = [
    "320193",    # Apple
    "789019",    # Microsoft
    "1018724",   # Amazon
    "1652044",   # Alphabet
    "1823584",   # Alliance Entertainment (AENT) — known escrow structure
    "1830081",   # RUM Group
]

# Filter to specific form types, or set to None to return everything
FORM_TYPES = ["10-K", "10-Q", "8-K"]

# Date range — both inclusive. Set either to None for an open bound.
START_DATE = "2025-04-30"   # "YYYY-MM-DD"
END_DATE   = "2026-04-30"   # "YYYY-MM-DD"

# Seconds to pause between SEC requests. 0.12 s ≈ 8 req/s (SEC cap is 10/s).
# Raise this (e.g. 0.5) if you ever see 429 rate-limit errors.
DELAY = 0.12

print(f"CIKs       : {CIKS}")
print(f"Form types : {FORM_TYPES}")
print(f"Date range : {START_DATE}  ->  {END_DATE}")
print(f"Delay      : {DELAY} s/request  (~{1/DELAY:.0f} req/s, SEC cap is 10 req/s)")

## 2. Fetch filing metadata

Processes each CIK one at a time with a `DELAY`-second pause between requests.
Progress is printed as each CIK completes.

In [ ]:
try:
    sync_filings = fetch_filings_for_ciks(
        ciks       = CIKS,
        form_types = FORM_TYPES,
        start_date = START_DATE,
        end_date   = END_DATE,
        delay      = DELAY,
    )
    print(f"\nTotal filings returned: {len(sync_filings)}")
except SECBlockedError as e:
    sync_filings = []
    print("\n" + "=" * 60)
    print("SEC TIMEOUT — rate limit exceeded.")
    print(str(e))
    print("Wait 10 minutes, then re-run from this cell.")
    print("=" * 60)

In [ ]:
# Show the filing metadata as a DataFrame
sync_df = pd.DataFrame([asdict(f) for f in sync_filings])
sync_df

## 3. Structured XBRL extraction (cover pages)

Parses the inline-XBRL instance for filings that have one, then splits the DEI
cover page into **entity** (one row per filing) and **security** (one row per
registered trading line) tables. 8-Ks and older non-inline-XBRL filings have no
instance and are skipped automatically.

In [ ]:
# Parse a few reports' XBRL cover pages (limit to keep the demo light).
xbrl_targets = sync_df[sync_df["form_type"].isin(["10-K", "10-Q", "20-F"])].head(3)
facts = parse_filings(xbrl_targets)
entity, security = cover_pages(facts)
entity

In [ ]:
security

## 4. Advanced / manual extraction (NEW)

`process_submission` downloads the **full submission text** (`submission_txt_url`),
then:

* **cleans** the primary document — strips the SGML envelope and inline-XBRL cruft
  into browser-renderable HTML, saved gzip-compressed as `*.clean.html.gz`;
* **extracts tables** tagged *geographic / revenue / segment / product*; and
* **flags concentration statements** (quantifier + subject + geography), e.g.
  *"The majority of our long-lived assets are located in China."*

It accepts a `sync_df` row, a `FilingRecord`, a dict, or a raw `.txt` URL.

In [ ]:
# Pick one annual report (10-K / 20-F / 40-F) to process in depth.
annual = sync_df[sync_df["form_type"].isin(["10-K", "20-F", "40-F"])]
target = annual.iloc[0] if len(annual) else sync_df.iloc[0]
print("processing:", target["ticker"], target["form_type"], target["filing_date"])

res = process_submission(
    target,                    # sync_df row / FilingRecord / dict / .txt URL
    save_dir = "clean_html",   # writes <ticker>_<form>_<date>_<acc>.clean.html.gz
    compress = True,
    delay    = DELAY,
)
print("primary document  :", res.primary_filename)
print("documents in .txt :", len(res.documents))
print("clean HTML chars  :", f"{len(res.clean_html):,}")
print("saved to          :", res.saved_path)
print("tables of interest:", len(res.tables))
print("statements found  :", len(res.statements))

In [ ]:
# Catalog of the tables we matched (geographic / revenue / segment / product).
res.catalog

In [ ]:
# The geographic tables — net sales / long-lived assets by country, if present.
# Use res.tables_for("revenue"|"segment"|"product") for the other categories.
geo_tables = res.tables_for("geographic")
print(f"{len(geo_tables)} geographic table(s)")
geo_tables[0] if geo_tables else "none found"

In [ ]:
# Geographic-concentration "footnote" statements found in the narrative.
# Filter to purely geographic ones with:  res.statements.dropna(subset=["geographies"])
res.statements

### 4b. Batch manual extraction

Run the manual extraction over every annual report in the fetch, sharing one
rate-limited session, then flatten the results into two tidy DataFrames:
`statements_frame` (all concentration statements) and `tables_frame` (all matched
tables — the parsed frame is kept in the `dataframe` column).

In [ ]:
# 8-K / 10-Q are skipped here to keep the download light — annual reports carry
# the geographic and segment disclosures we care about.
annuals = sync_df[sync_df["form_type"].isin(["10-K", "20-F", "40-F"])]
results = process_filings(annuals, save_dir="clean_html", compress=True, delay=DELAY)

all_statements = statements_frame(results)   # one flat DataFrame across filings
all_tables     = tables_frame(results)       # one flat table catalog
print(f"\n{len(results)} filings processed | "
      f"{len(all_statements)} statements | {len(all_tables)} tables")

In [ ]:
all_statements

In [ ]:
# Table catalog across all filings (drop the DataFrame objects for a clean view).
all_tables.drop(columns=["dataframe"])

---
### Notes & tips

* **Saved HTML** lands in `clean_html/` next to this notebook, gzip-compressed.
  Open one with `import gzip; print(gzip.open(path, "rt", encoding="utf-8").read())`,
  or gunzip it and open in a browser.
* **Rate limits** — everything obeys the SEC ≤10 req/s cap; a `SECBlockedError`
  means a 10-minute IP cool-off, so stop and wait before re-running.
* **Filtering statements** — `all_statements[all_statements.category == "assets"]`
  for asset-location disclosures, or `.dropna(subset=["geographies"])` for the ones
  that name a specific country/region.
* **Table categories** — pass `categories=(...)` to `extract_tables`, or call
  `res.tables_for("revenue"|"segment"|"product"|"geographic")`.
* **40-F wrappers** (some Canadian filers) carry the real annual report as a
  separate exhibit, so the primary document — and its table count — may be thin.

---
## 5. Country-assignment monthly monitor (NEW)

`run_monthly` pulls each company's **current** country of incorporation + HQ from
the EDGAR *submissions* API (the authoritative, real-time source), then validates
it against the latest cover-bearing filing — including **6-K / non-XBRL** forms,
and covers that carry only a principal office. It flags **dual-HQ**,
**ISO-collision** codes (e.g. `CA`=California, not Canada), and
**profile-vs-filing mismatches** (e.g. a profile that says `K3`=Hong Kong while
the cover says Cayman Islands).

Each run writes a timestamped snapshot to `SNAP_DIR` and **diffs** it against the
previous month's snapshot, so month-over-month changes/updates surface
automatically. (Compare the snapshot against your own database separately.)

> For a whole-market run, set `validate_with_filing=False` for a fast,
> profile-only pass, or pull the nightly `submissions.zip` bulk file and use
> `country_assignment.iter_bulk_profiles` instead of per-CIK requests.

In [ ]:
from methods.country_assignment import run_monthly

# Reuse the Inputs CIKs (or set your own list). Each run writes
# country_assignment_<YYYY-MM>.csv + changes_<YYYY-MM>.csv under SNAP_DIR.
SNAP_DIR = "country_assignment_snapshots"

snapshot, changes = run_monthly(
    CIKS,
    out_dir              = SNAP_DIR,
    # month              = "2026-07",   # defaults to the current calendar month
    validate_with_filing = True,        # False = fast, profile-only (whole-market runs)
    delay                = DELAY,
)

In [ ]:
# This month's assignments (also saved to country_assignment_<YYYY-MM>.csv).
snapshot[[
    "ticker", "entity_name", "incorporation_code", "incorporation_name",
    "incorporation_country", "incorporation_agreement", "hq_country",
    "hq_source", "dual_hq", "latest_form", "status",
]]

In [ ]:
# Only the rows that need a look — mismatch / dual-HQ / uncorroborated ISO trap —
# with the human-readable reasons in `issues`.
review = snapshot[snapshot["status"] == "review"][[
    "ticker", "incorporation_code", "incorporation_country", "hq_country",
    "dual_hq", "iso_trap", "issues",
]]
review

In [ ]:
# Field-level changes vs last month's snapshot (empty on the very first run).
# Re-run next month and this fills in from the saved snapshots:
#   change_type ∈ {added, removed, changed};  field/old_value/new_value per change.
changes

## 6. Derived facts — ICB / voting rights / ADR ratios (`analysis/`)

The `analysis` package derives facts that XBRL doesn't carry, using
**deterministic rules first and a local Ollama LLM second** (the model only
interprets text it is shown, and every result records method, model,
confidence and the evidence sentence).

- `adr_ratio` — underlying shares per ADS (cover title → filing prose → LLM)
- `voting_rights` — votes per share by class (rules + LLM over the same sentences)
- `icb_classify` — ICB industry/sector/subsector (SIC prior + LLM, enum by name)

The LLM is optional and has two backends: an **Ollama server**, or the
**embedded** in-process model — its weights file lives at
`analysis/models/gemma-2-2b-it-Q4_K_M.gguf` (~1.7 GB, gitignored;
re-fetch with `python analysis/models/download_model.py`). Env var
`LLM_BACKEND` picks (`auto`/`ollama`/`embedded`/`none`); without either
backend the deterministic layers still run.

In [ ]:
# Offline demo: the deterministic layers need no network and no LLM.
from analysis.adr_ratio import extract_adr_ratio
from analysis.voting_rights import extract_voting_rights
from analysis.ollama_client import backend_info

r = extract_adr_ratio(title="American Depositary Shares, each representing "
                            "two (2) ordinary shares", use_llm=False)
print("ADR :", r.ratio_display, "| via", r.method)

v = extract_voting_rights(
    "Our Class B common stock has ten votes per share and our Class A "
    "common stock has one vote per share.", use_llm=False)
for x in v:
    print("VOTE:", x.class_label, "=", x.votes_per_share)

info = backend_info()   # ollama server, embedded .gguf, or None
print("LLM backend:", info["backend"] or "none (deterministic layers only)",
      "| model:", info["default"])

In [ ]:
# Full pipeline for one company: fetches the latest annual report, derives
# all three fact kinds, and stores them (with evidence) for analyst review.
# Takes ~1 minute per company on first run; results land in
# analysis/research.sqlite and in the viewer's /review queue.
from analysis.analyze import analyze_company

result = analyze_company("SHEL")     # ticker or CIK
{k: v for k, v in result.items() if k in ("name", "icb", "voting", "adr")}

In [ ]:
# What's in the research store, and where each fact stands with reviewers:
import pandas as pd
from analysis.research_store import ResearchDB

db = ResearchDB()
overview = pd.DataFrame(db.companies())
recent = pd.DataFrame([{ "ticker": r["ticker"], "kind": r["kind"],
                         "confidence": r["confidence"], "status": r["status"],
                         "source": f"{r['source_form']} {r['source_filing_date']}"}
                       for r in db.recent(12)])
db.close()
print(db.__class__.__name__, "->", "analysis/research.sqlite")
display(overview.head(10)); display(recent)